### Transformar Datos de "Employees" como JSON String
1. Extraer valores de la columna de nivel superior
2. Extraer elementos de la matriz(arrays)
3. Extraer valores de columnas anidadas
4. Convierte valores de columna a un tipo de dato especifico

In [0]:
df_employees = spark.table("zenviro.bronze.employees_py")
display(df_employees)

#### 1. Extraer valores de la columna de nivel superior

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

schema = StructType([
    StructField("employee_id", IntegerType(), True)
])

In [0]:
from pyspark.sql.functions import col, from_json

df_column_value = (
    df_employees
    .withColumn("value_parsed", from_json(col("value"), schema))
    .select(
        col("value_parsed.employee_id"),
        col("value")
    )
)
display(df_column_value)

#### 2. Extraer elementos de la matriz(arrays)

In [0]:
from pyspark.sql.types import StructType, StructField, ArrayType, StringType

schema = StructType([
    StructField("name", ArrayType(StringType()), True)
])

In [0]:
from pyspark.sql.functions import col, from_json, get

df_array_value = (
    df_employees
    .withColumn("value_parsed", from_json(col("value"), schema))
    .select(
        get(col("value_parsed.name"), 0).alias("name_0"),
        get(col("value_parsed.name"), 1).alias("name_1"),
        col("value")
    )
)
display(df_array_value)

#### 3. Extraer valores de columnas anidadas

In [0]:
from pyspark.sql.types import StructType, StructField, ArrayType, StringType

schema = StructType([
    StructField(
        "name", 
        ArrayType(
            StructType([
                StructField("first_name", StringType(), True),
                StructField("last_name", StringType(), True)
            ])
        ),
        True
    )
])

In [0]:
from pyspark.sql.functions import col, from_json, get

df_array_column_value = (
    df_employees
    .withColumn("value_parsed", from_json(col("value"), schema))
    .select(
        get(col("value_parsed.name"), 0).getField("first_name").alias("first_name"),
        get(col("value_parsed.name"), 0).getField("last_name").alias("last_name"),
        col("value")
    )
)
display(df_array_column_value)

#### 4. Convierte valores de columna a un tipo de dato especifico

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, ArrayType, StringType

schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("birth_date", DateType(), True),
    StructField(
        "name", 
        ArrayType(
            StructType([
                StructField("first_name", StringType(), True),
                StructField("last_name", StringType(), True)
            ])
        ),
        True
    )
])

In [0]:
from pyspark.sql.functions import col, from_json, get

df_data_type_column_value = (
    df_employees
    .withColumn("value_parsed", from_json(col("value"), schema))
    .select(
        col("value_parsed.employee_id"),
        col("value_parsed.birth_date"),
        get(col("value_parsed.name"), 0).getField("first_name").alias("first_name"),
        get(col("value_parsed.name"), 0).getField("last_name").alias("last_name"),
        col("value")
    )
)
display(df_data_type_column_value)